In [1]:
import pandas as pd
import re

# =====================================================
# LOAD SOUTHERN STATION DATA
# =====================================================

INPUT_FILE = "/content/southern_wordplay_full_v1.csv"

stations = pd.read_csv(INPUT_FILE)

print("Loaded station file")
print(f"Rows: {len(stations)}")
print()

# =====================================================
# STANDARDISE COLUMN NAMES
# =====================================================

stations.columns = [c.strip().lower() for c in stations.columns]

print("Columns found:")
print(list(stations.columns))
print()

# =====================================================
# FIND STATION COLUMN
# =====================================================

if "station" in stations.columns:
    station_col = "station"
elif "station_name" in stations.columns:
    station_col = "station_name"
else:
    raise ValueError(
        "Could not find station column. "
        "Expected 'station' or 'station_name'."
    )

print(f"Using station column: {station_col}")
print()

# =====================================================
# BUILD STATION DIMENSION
# =====================================================

dim_stations = (
    stations[[station_col]]
    .drop_duplicates()
    .rename(columns={station_col: "station"})
)

# =====================================================
# GENERATE STATION IDS
# =====================================================

def make_station_id(name):
    clean = re.sub(r"[^a-z0-9]+", "_", str(name).lower())
    clean = clean.strip("_")
    return f"STN_{clean}"

dim_stations["station_id"] = (
    dim_stations["station"]
    .apply(make_station_id)
)

# =====================================================
# REORDER COLUMNS
# =====================================================

dim_stations = dim_stations[
    ["station_id", "station"]
]

print("Station dimension created")
print(f"Unique stations: {len(dim_stations)}")
print()

# =====================================================
# CURATED SOUTHERN LANDMARKS
# =====================================================

seed_landmarks = [

    # -------------------------------------------------
    # BRIGHTON
    # -------------------------------------------------

    {
        "station": "Brighton",
        "landmark": "Brighton Palace Pier",
        "distance_km": 0.7,
        "proximity": "Walkable",
    },

    {
        "station": "Brighton",
        "landmark": "Royal Pavilion",
        "distance_km": 0.5,
        "proximity": "Walkable",
    },

    # -------------------------------------------------
    # LEWES
    # -------------------------------------------------

    {
        "station": "Lewes",
        "landmark": "Lewes Castle",
        "distance_km": 0.6,
        "proximity": "Walkable",
    },

    # -------------------------------------------------
    # ARUNDEL
    # -------------------------------------------------

    {
        "station": "Arundel",
        "landmark": "Arundel Castle",
        "distance_km": 0.8,
        "proximity": "Walkable",
    },

    # -------------------------------------------------
    # HASTINGS
    # -------------------------------------------------

    {
        "station": "Hastings",
        "landmark": "Smugglers Adventure",
        "distance_km": 0.9,
        "proximity": "Walkable",
    },

    # -------------------------------------------------
    # PORTSMOUTH HARBOUR
    # -------------------------------------------------

    {
        "station": "Portsmouth Harbour",
        "landmark": "Historic Dockyard",
        "distance_km": 0.3,
        "proximity": "Next Door",
    },

    # -------------------------------------------------
    # EASTBOURNE
    # -------------------------------------------------

    {
        "station": "Eastbourne",
        "landmark": "Eastbourne Pier",
        "distance_km": 1.0,
        "proximity": "Walkable",
    },

    # -------------------------------------------------
    # CHICHESTER
    # -------------------------------------------------

    {
        "station": "Chichester",
        "landmark": "Chichester Cathedral",
        "distance_km": 0.7,
        "proximity": "Walkable",
    },

    # -------------------------------------------------
    # RYE
    # -------------------------------------------------

    {
        "station": "Rye",
        "landmark": "Mermaid Street",
        "distance_km": 0.4,
        "proximity": "Walkable",
    },

    # -------------------------------------------------
    # CRYSTAL PALACE
    # -------------------------------------------------

    {
        "station": "Crystal Palace",
        "landmark": "Crystal Palace Park",
        "distance_km": 0.8,
        "proximity": "Walkable",
    },

    # -------------------------------------------------
    # BOGNOR REGIS
    # -------------------------------------------------

    {
        "station": "Bognor Regis",
        "landmark": "Bognor Regis Beach",
        "distance_km": 0.5,
        "proximity": "Walkable",
    },

    # -------------------------------------------------
    # WORTHING
    # -------------------------------------------------

    {
        "station": "Worthing",
        "landmark": "Worthing Pier",
        "distance_km": 0.6,
        "proximity": "Walkable",
    },

    # -------------------------------------------------
    # SHOREHAM-BY-SEA
    # -------------------------------------------------

    {
        "station": "Shoreham-by-Sea",
        "landmark": "Shoreham Airport",
        "distance_km": 2.0,
        "proximity": "Nearby",
    },

    # -------------------------------------------------
    # HAMPTON COURT
    # -------------------------------------------------

    {
        "station": "Hampton Court",
        "landmark": "Hampton Court Palace",
        "distance_km": 0.3,
        "proximity": "Next Door",
    },

]

# =====================================================
# BUILD LANDMARK FACT TABLE
# =====================================================

fact_landmarks = pd.DataFrame(seed_landmarks)

# =====================================================
# GENERATE LANDMARK IDS
# =====================================================

def make_landmark_id(name):
    clean = re.sub(r"[^a-z0-9]+", str(name).lower())
    clean = clean.strip("_")
    return f"LMK_{clean}"

fact_landmarks["landmark_id"] = (
    fact_landmarks["landmark"]
    .apply(make_landmark_id)
)

# =====================================================
# JOIN STATION IDS
# =====================================================

fact_landmarks = fact_landmarks.merge(
    dim_stations,
    on="station",
    how="left"
)

# =====================================================
# VALIDATION
# =====================================================

missing = fact_landmarks[
    fact_landmarks["station_id"].isna()
]

print("Validation")
print("----------")

if len(missing) > 0:

    print()
    print("WARNING: Missing station matches")
    print()

    print(
        missing[
            ["station", "landmark"]
        ]
    )

else:

    print("All stations matched successfully")

print()

# =====================================================
# FINAL COLUMN ORDER
# =====================================================

fact_landmarks = fact_landmarks[
    [
        "station_id",
        "station",
        "landmark_id",
        "landmark",
        "distance_km",
        "proximity",
    ]
]

# =====================================================
# EXPORT CSV FILES
# =====================================================

DIM_OUTPUT = "/content/dim_southern_stations.csv"

FACT_OUTPUT = "/content/fact_southern_station_landmark.csv"

dim_stations.to_csv(
    DIM_OUTPUT,
    index=False
)

fact_landmarks.to_csv(
    FACT_OUTPUT,
    index=False
)

# =====================================================
# PREVIEW OUTPUTS
# =====================================================

print("DONE")
print()

print("Created files:")
print(DIM_OUTPUT)
print(FACT_OUTPUT)

print()
print("Station dimension preview")
print("-------------------------")
print(dim_stations.head())

print()
print("Landmark fact preview")
print("---------------------")
print(fact_landmarks.head(20))

print()
print(f"Total stations: {len(dim_stations)}")
print(f"Total landmarks: {len(fact_landmarks)}")

Loaded station file
Rows: 107

Columns found:
['station', 'station_name', 'route_count', 'service_count', 'service_density', 'route_diversity_band', 'is_terminus', 'is_interchange', 'time_from_london', 'time_band', 'accessibility_score', 'difficulty_score', 'name_lower', 'word_count', 'has_hyphen', 'has_apostrophe', 'has_brackets', 'has_ampersand', 'has_double_letter', 'has_alliteration', 'compass_direction', 'has_direction', 'has_saint', 'initial_letter', 'name_shape', 'suffix', 'is_coastal', 'wordplay_bucket', 'letters_only', 'char_count', 'vowel_count', 'consonant_count', 'unique_letters', 'starts_with_vowel', 'ends_with_vowel', 'is_long_name', 'has_rare_word', 'pattern_tags', 'wordplay_difficulty', 'wordplay_score']

Using station column: station

Station dimension created
Unique stations: 81



TypeError: sub() missing 1 required positional argument: 'string'

In [2]:
import pandas as pd
import re

# =====================================================
# LOAD SOUTHERN STATION DATA
# =====================================================

INPUT_FILE = "/content/southern_wordplay_full_v1.csv"

stations = pd.read_csv(INPUT_FILE)

print("Loaded station file")
print(f"Rows: {len(stations)}")
print()

# =====================================================
# STANDARDISE COLUMN NAMES
# =====================================================

stations.columns = [c.strip().lower() for c in stations.columns]

print("Columns found:")
print(list(stations.columns))
print()

# =====================================================
# FIND STATION COLUMN
# =====================================================

if "station" in stations.columns:
    station_col = "station"

elif "station_name" in stations.columns:
    station_col = "station_name"

else:
    raise ValueError(
        "Could not find station column. "
        "Expected 'station' or 'station_name'."
    )

print(f"Using station column: {station_col}")
print()

# =====================================================
# BUILD STATION DIMENSION
# =====================================================

dim_stations = (
    stations[[station_col]]
    .drop_duplicates()
    .rename(columns={station_col: "station"})
)

# =====================================================
# GENERATE STATION IDS
# =====================================================

def make_station_id(name):

    clean = re.sub(
        r"[^a-z0-9]+",
        "_",
        str(name).lower()
    )

    clean = clean.strip("_")

    return f"STN_{clean}"

dim_stations["station_id"] = (
    dim_stations["station"]
    .apply(make_station_id)
)

# =====================================================
# REORDER COLUMNS
# =====================================================

dim_stations = dim_stations[
    ["station_id", "station"]
]

print("Station dimension created")
print(f"Unique stations: {len(dim_stations)}")
print()

# =====================================================
# CURATED SOUTHERN LANDMARKS
# =====================================================

seed_landmarks = [

    # -------------------------------------------------
    # BRIGHTON
    # -------------------------------------------------

    {
        "station": "Brighton",
        "landmark": "Brighton Palace Pier",
        "distance_km": 0.7,
        "proximity": "Walkable",
    },

    {
        "station": "Brighton",
        "landmark": "Royal Pavilion",
        "distance_km": 0.5,
        "proximity": "Walkable",
    },

    # -------------------------------------------------
    # LEWES
    # -------------------------------------------------

    {
        "station": "Lewes",
        "landmark": "Lewes Castle",
        "distance_km": 0.6,
        "proximity": "Walkable",
    },

    # -------------------------------------------------
    # ARUNDEL
    # -------------------------------------------------

    {
        "station": "Arundel",
        "landmark": "Arundel Castle",
        "distance_km": 0.8,
        "proximity": "Walkable",
    },

    # -------------------------------------------------
    # HASTINGS
    # -------------------------------------------------

    {
        "station": "Hastings",
        "landmark": "Smugglers Adventure",
        "distance_km": 0.9,
        "proximity": "Walkable",
    },

    # -------------------------------------------------
    # PORTSMOUTH HARBOUR
    # -------------------------------------------------

    {
        "station": "Portsmouth Harbour",
        "landmark": "Historic Dockyard",
        "distance_km": 0.3,
        "proximity": "Next Door",
    },

    # -------------------------------------------------
    # EASTBOURNE
    # -------------------------------------------------

    {
        "station": "Eastbourne",
        "landmark": "Eastbourne Pier",
        "distance_km": 1.0,
        "proximity": "Walkable",
    },

    # -------------------------------------------------
    # CHICHESTER
    # -------------------------------------------------

    {
        "station": "Chichester",
        "landmark": "Chichester Cathedral",
        "distance_km": 0.7,
        "proximity": "Walkable",
    },

    # -------------------------------------------------
    # RYE
    # -------------------------------------------------

    {
        "station": "Rye",
        "landmark": "Mermaid Street",
        "distance_km": 0.4,
        "proximity": "Walkable",
    },

    # -------------------------------------------------
    # CRYSTAL PALACE
    # -------------------------------------------------

    {
        "station": "Crystal Palace",
        "landmark": "Crystal Palace Park",
        "distance_km": 0.8,
        "proximity": "Walkable",
    },

    # -------------------------------------------------
    # BOGNOR REGIS
    # -------------------------------------------------

    {
        "station": "Bognor Regis",
        "landmark": "Bognor Regis Beach",
        "distance_km": 0.5,
        "proximity": "Walkable",
    },

    # -------------------------------------------------
    # WORTHING
    # -------------------------------------------------

    {
        "station": "Worthing",
        "landmark": "Worthing Pier",
        "distance_km": 0.6,
        "proximity": "Walkable",
    },

    # -------------------------------------------------
    # SHOREHAM-BY-SEA
    # -------------------------------------------------

    {
        "station": "Shoreham-by-Sea",
        "landmark": "Shoreham Airport",
        "distance_km": 2.0,
        "proximity": "Nearby",
    },

    # -------------------------------------------------
    # HAMPTON COURT
    # -------------------------------------------------

    {
        "station": "Hampton Court",
        "landmark": "Hampton Court Palace",
        "distance_km": 0.3,
        "proximity": "Next Door",
    },

]

# =====================================================
# BUILD LANDMARK FACT TABLE
# =====================================================

fact_landmarks = pd.DataFrame(seed_landmarks)

# =====================================================
# GENERATE LANDMARK IDS
# =====================================================

def make_landmark_id(name):

    clean = re.sub(
        r"[^a-z0-9]+",
        "_",
        str(name).lower()
    )

    clean = clean.strip("_")

    return f"LMK_{clean}"

fact_landmarks["landmark_id"] = (
    fact_landmarks["landmark"]
    .apply(make_landmark_id)
)

# =====================================================
# JOIN STATION IDS
# =====================================================

fact_landmarks = fact_landmarks.merge(
    dim_stations,
    on="station",
    how="left"
)

# =====================================================
# VALIDATION
# =====================================================

missing = fact_landmarks[
    fact_landmarks["station_id"].isna()
]

print("Validation")
print("----------")

if len(missing) > 0:

    print()
    print("WARNING: Missing station matches")
    print()

    print(
        missing[
            ["station", "landmark"]
        ]
    )

else:

    print("All stations matched successfully")

print()

# =====================================================
# FINAL COLUMN ORDER
# =====================================================

fact_landmarks = fact_landmarks[
    [
        "station_id",
        "station",
        "landmark_id",
        "landmark",
        "distance_km",
        "proximity",
    ]
]

# =====================================================
# EXPORT CSV FILES
# =====================================================

DIM_OUTPUT = "/content/dim_southern_stations.csv"

FACT_OUTPUT = "/content/fact_southern_station_landmark.csv"

dim_stations.to_csv(
    DIM_OUTPUT,
    index=False
)

fact_landmarks.to_csv(
    FACT_OUTPUT,
    index=False
)

# =====================================================
# PREVIEW OUTPUTS
# =====================================================

print("DONE")
print()

print("Created files:")
print(DIM_OUTPUT)
print(FACT_OUTPUT)

print()
print("Station dimension preview")
print("-------------------------")
print(dim_stations.head())

print()
print("Landmark fact preview")
print("---------------------")
print(fact_landmarks.head(20))

print()
print(f"Total stations: {len(dim_stations)}")
print(f"Total landmarks: {len(fact_landmarks)}")

Loaded station file
Rows: 107

Columns found:
['station', 'station_name', 'route_count', 'service_count', 'service_density', 'route_diversity_band', 'is_terminus', 'is_interchange', 'time_from_london', 'time_band', 'accessibility_score', 'difficulty_score', 'name_lower', 'word_count', 'has_hyphen', 'has_apostrophe', 'has_brackets', 'has_ampersand', 'has_double_letter', 'has_alliteration', 'compass_direction', 'has_direction', 'has_saint', 'initial_letter', 'name_shape', 'suffix', 'is_coastal', 'wordplay_bucket', 'letters_only', 'char_count', 'vowel_count', 'consonant_count', 'unique_letters', 'starts_with_vowel', 'ends_with_vowel', 'is_long_name', 'has_rare_word', 'pattern_tags', 'wordplay_difficulty', 'wordplay_score']

Using station column: station

Station dimension created
Unique stations: 81

Validation
----------


               station              landmark
0             Brighton  Brighton Palace Pier
1             Brighton        Royal Pavilion
2                Lewes          

In [3]:
import pandas as pd
import re

# =====================================================
# LOAD SOUTHERN STATION DATA
# =====================================================

INPUT_FILE = "/content/southern_wordplay_full_v1.csv"

stations = pd.read_csv(INPUT_FILE)

print("Loaded station file")
print(f"Rows: {len(stations)}")
print()

# =====================================================
# STANDARDISE COLUMN NAMES
# =====================================================

stations.columns = [c.strip().lower() for c in stations.columns]

print("Columns found:")
print(list(stations.columns))
print()

# =====================================================
# USE HUMAN-READABLE STATION NAMES
# =====================================================

station_col = "station_name"

if station_col not in stations.columns:
    raise ValueError(
        "Could not find 'station_name' column."
    )

print(f"Using station column: {station_col}")
print()

# =====================================================
# BUILD STATION DIMENSION
# =====================================================

dim_stations = (
    stations[["station", station_col]]
    .drop_duplicates()
    .rename(columns={
        "station": "station_code",
        station_col: "station"
    })
)

# =====================================================
# GENERATE STATION IDS
# =====================================================

def make_station_id(name):

    clean = re.sub(
        r"[^a-z0-9]+",
        "_",
        str(name).lower()
    )

    clean = clean.strip("_")

    return f"STN_{clean}"

dim_stations["station_id"] = (
    dim_stations["station"]
    .apply(make_station_id)
)

# =====================================================
# REORDER COLUMNS
# =====================================================

dim_stations = dim_stations[
    [
        "station_id",
        "station",
        "station_code",
    ]
]

print("Station dimension created")
print(f"Unique stations: {len(dim_stations)}")
print()

# =====================================================
# CURATED SOUTHERN LANDMARKS
# =====================================================

seed_landmarks = [

    # -------------------------------------------------
    # BRIGHTON
    # -------------------------------------------------

    {
        "station": "Brighton",
        "landmark": "Brighton Palace Pier",
        "distance_km": 0.7,
        "proximity": "Walkable",
    },

    {
        "station": "Brighton",
        "landmark": "Royal Pavilion",
        "distance_km": 0.5,
        "proximity": "Walkable",
    },

    # -------------------------------------------------
    # LEWES
    # -------------------------------------------------

    {
        "station": "Lewes",
        "landmark": "Lewes Castle",
        "distance_km": 0.6,
        "proximity": "Walkable",
    },

    # -------------------------------------------------
    # ARUNDEL
    # -------------------------------------------------

    {
        "station": "Arundel",
        "landmark": "Arundel Castle",
        "distance_km": 0.8,
        "proximity": "Walkable",
    },

    # -------------------------------------------------
    # HASTINGS
    # -------------------------------------------------

    {
        "station": "Hastings",
        "landmark": "Smugglers Adventure",
        "distance_km": 0.9,
        "proximity": "Walkable",
    },

    # -------------------------------------------------
    # PORTSMOUTH HARBOUR
    # -------------------------------------------------

    {
        "station": "Portsmouth Harbour",
        "landmark": "Historic Dockyard",
        "distance_km": 0.3,
        "proximity": "Next Door",
    },

    # -------------------------------------------------
    # EASTBOURNE
    # -------------------------------------------------

    {
        "station": "Eastbourne",
        "landmark": "Eastbourne Pier",
        "distance_km": 1.0,
        "proximity": "Walkable",
    },

    # -------------------------------------------------
    # CHICHESTER
    # -------------------------------------------------

    {
        "station": "Chichester",
        "landmark": "Chichester Cathedral",
        "distance_km": 0.7,
        "proximity": "Walkable",
    },

    # -------------------------------------------------
    # RYE
    # -------------------------------------------------

    {
        "station": "Rye",
        "landmark": "Mermaid Street",
        "distance_km": 0.4,
        "proximity": "Walkable",
    },

    # -------------------------------------------------
    # CRYSTAL PALACE
    # -------------------------------------------------

    {
        "station": "Crystal Palace",
        "landmark": "Crystal Palace Park",
        "distance_km": 0.8,
        "proximity": "Walkable",
    },

    # -------------------------------------------------
    # BOGNOR REGIS
    # -------------------------------------------------

    {
        "station": "Bognor Regis",
        "landmark": "Bognor Regis Beach",
        "distance_km": 0.5,
        "proximity": "Walkable",
    },

    # -------------------------------------------------
    # WORTHING
    # -------------------------------------------------

    {
        "station": "Worthing",
        "landmark": "Worthing Pier",
        "distance_km": 0.6,
        "proximity": "Walkable",
    },

    # -------------------------------------------------
    # SHOREHAM-BY-SEA
    # -------------------------------------------------

    {
        "station": "Shoreham-by-Sea",
        "landmark": "Shoreham Airport",
        "distance_km": 2.0,
        "proximity": "Nearby",
    },

    # -------------------------------------------------
    # HAMPTON COURT
    # -------------------------------------------------

    {
        "station": "Hampton Court",
        "landmark": "Hampton Court Palace",
        "distance_km": 0.3,
        "proximity": "Next Door",
    },

]

# =====================================================
# BUILD LANDMARK FACT TABLE
# =====================================================

fact_landmarks = pd.DataFrame(seed_landmarks)

# =====================================================
# GENERATE LANDMARK IDS
# =====================================================

def make_landmark_id(name):

    clean = re.sub(
        r"[^a-z0-9]+",
        "_",
        str(name).lower()
    )

    clean = clean.strip("_")

    return f"LMK_{clean}"

fact_landmarks["landmark_id"] = (
    fact_landmarks["landmark"]
    .apply(make_landmark_id)
)

# =====================================================
# JOIN STATION IDS
# =====================================================

fact_landmarks = fact_landmarks.merge(
    dim_stations,
    on="station",
    how="left"
)

# =====================================================
# VALIDATION
# =====================================================

missing = fact_landmarks[
    fact_landmarks["station_id"].isna()
]

print("Validation")
print("----------")

if len(missing) > 0:

    print()
    print("WARNING: Missing station matches")
    print()

    print(
        missing[
            ["station", "landmark"]
        ]
    )

else:

    print("All stations matched successfully")

print()

# =====================================================
# FINAL COLUMN ORDER
# =====================================================

fact_landmarks = fact_landmarks[
    [
        "station_id",
        "station",
        "station_code",
        "landmark_id",
        "landmark",
        "distance_km",
        "proximity",
    ]
]

# =====================================================
# EXPORT CSV FILES
# =====================================================

DIM_OUTPUT = "/content/dim_southern_stations.csv"

FACT_OUTPUT = "/content/fact_southern_station_landmark.csv"

dim_stations.to_csv(
    DIM_OUTPUT,
    index=False
)

fact_landmarks.to_csv(
    FACT_OUTPUT,
    index=False
)

# =====================================================
# PREVIEW OUTPUTS
# =====================================================

print("DONE")
print()

print("Created files:")
print(DIM_OUTPUT)
print(FACT_OUTPUT)

print()
print("Station dimension preview")
print("-------------------------")
print(dim_stations.head())

print()
print("Landmark fact preview")
print("---------------------")
print(fact_landmarks.head(20))

print()
print(f"Total stations: {len(dim_stations)}")
print(f"Total landmarks: {len(fact_landmarks)}")

Loaded station file
Rows: 107

Columns found:
['station', 'station_name', 'route_count', 'service_count', 'service_density', 'route_diversity_band', 'is_terminus', 'is_interchange', 'time_from_london', 'time_band', 'accessibility_score', 'difficulty_score', 'name_lower', 'word_count', 'has_hyphen', 'has_apostrophe', 'has_brackets', 'has_ampersand', 'has_double_letter', 'has_alliteration', 'compass_direction', 'has_direction', 'has_saint', 'initial_letter', 'name_shape', 'suffix', 'is_coastal', 'wordplay_bucket', 'letters_only', 'char_count', 'vowel_count', 'consonant_count', 'unique_letters', 'starts_with_vowel', 'ends_with_vowel', 'is_long_name', 'has_rare_word', 'pattern_tags', 'wordplay_difficulty', 'wordplay_score']

Using station column: station_name

Station dimension created
Unique stations: 107

Validation
----------


           station              landmark
3          Arundel        Arundel Castle
8              Rye        Mermaid Street
9   Crystal Palace   Crystal Palace Pa

In [4]:
import pandas as pd
import re

# =====================================================
# LOAD SOUTHERN STATION DATA
# =====================================================

INPUT_FILE = "/content/southern_wordplay_full_v1.csv"

stations = pd.read_csv(INPUT_FILE)

print("Loaded station file")
print(f"Rows: {len(stations)}")
print()

# =====================================================
# STANDARDISE COLUMN NAMES
# =====================================================

stations.columns = [c.strip().lower() for c in stations.columns]

print("Columns found:")
print(list(stations.columns))
print()

# =====================================================
# USE HUMAN-READABLE STATION NAMES
# =====================================================

station_col = "station_name"

if station_col not in stations.columns:
    raise ValueError(
        "Could not find 'station_name' column."
    )

print(f"Using station column: {station_col}")
print()

# =====================================================
# BUILD STATION DIMENSION
# =====================================================

dim_stations = (
    stations[["station", station_col]]
    .drop_duplicates()
    .rename(columns={
        "station": "station_code",
        station_col: "station"
    })
)

# =====================================================
# GENERATE STATION IDS
# =====================================================

def make_station_id(name):

    clean = re.sub(
        r"[^a-z0-9]+",
        "_",
        str(name).lower()
    )

    clean = clean.strip("_")

    return f"STN_{clean}"

dim_stations["station_id"] = (
    dim_stations["station"]
    .apply(make_station_id)
)

# =====================================================
# REORDER COLUMNS
# =====================================================

dim_stations = dim_stations[
    [
        "station_id",
        "station",
        "station_code",
    ]
]

print("Station dimension created")
print(f"Unique stations: {len(dim_stations)}")
print()

# =====================================================
# CURATED SOUTHERN LANDMARKS
# =====================================================

seed_landmarks = [

    # -------------------------------------------------
    # BRIGHTON
    # -------------------------------------------------

    {
        "station": "Brighton",
        "landmark": "Brighton Palace Pier",
        "distance_km": 0.7,
        "proximity": "Walkable",
    },

    {
        "station": "Brighton",
        "landmark": "Royal Pavilion",
        "distance_km": 0.5,
        "proximity": "Walkable",
    },

    # -------------------------------------------------
    # LEWES
    # -------------------------------------------------

    {
        "station": "Lewes",
        "landmark": "Lewes Castle",
        "distance_km": 0.6,
        "proximity": "Walkable",
    },

    # -------------------------------------------------
    # HASTINGS
    # -------------------------------------------------

    {
        "station": "Hastings",
        "landmark": "Smugglers Adventure",
        "distance_km": 0.9,
        "proximity": "Walkable",
    },

    # -------------------------------------------------
    # PORTSMOUTH HARBOUR
    # -------------------------------------------------

    {
        "station": "Portsmouth Harbour",
        "landmark": "Historic Dockyard",
        "distance_km": 0.3,
        "proximity": "Next Door",
    },

    # -------------------------------------------------
    # EASTBOURNE
    # -------------------------------------------------

    {
        "station": "Eastbourne",
        "landmark": "Eastbourne Pier",
        "distance_km": 1.0,
        "proximity": "Walkable",
    },

    # -------------------------------------------------
    # CHICHESTER
    # -------------------------------------------------

    {
        "station": "Chichester",
        "landmark": "Chichester Cathedral",
        "distance_km": 0.7,
        "proximity": "Walkable",
    },

    # -------------------------------------------------
    # WORTHING
    # -------------------------------------------------

    {
        "station": "Worthing",
        "landmark": "Worthing Pier",
        "distance_km": 0.6,
        "proximity": "Walkable",
    },

    # -------------------------------------------------
    # SHOREHAM-BY-SEA
    # -------------------------------------------------

    {
        "station": "Shoreham-by-Sea",
        "landmark": "Shoreham Airport",
        "distance_km": 2.0,
        "proximity": "Nearby",
    },

]

# =====================================================
# BUILD LANDMARK FACT TABLE
# =====================================================

fact_landmarks = pd.DataFrame(seed_landmarks)

# =====================================================
# GENERATE LANDMARK IDS
# =====================================================

def make_landmark_id(name):

    clean = re.sub(
        r"[^a-z0-9]+",
        "_",
        str(name).lower()
    )

    clean = clean.strip("_")

    return f"LMK_{clean}"

fact_landmarks["landmark_id"] = (
    fact_landmarks["landmark"]
    .apply(make_landmark_id)
)

# =====================================================
# JOIN STATION IDS
# =====================================================

fact_landmarks = fact_landmarks.merge(
    dim_stations,
    on="station",
    how="left"
)

# =====================================================
# VALIDATION
# =====================================================

missing = fact_landmarks[
    fact_landmarks["station_id"].isna()
]

print("Validation")
print("----------")

if len(missing) > 0:

    print()
    print("WARNING: Missing station matches")
    print()

    print(
        missing[
            ["station", "landmark"]
        ]
    )

else:

    print("All stations matched successfully")

print()

# =====================================================
# FINAL COLUMN ORDER
# =====================================================

fact_landmarks = fact_landmarks[
    [
        "station_id",
        "station",
        "station_code",
        "landmark_id",
        "landmark",
        "distance_km",
        "proximity",
    ]
]

# =====================================================
# EXPORT CSV FILES
# =====================================================

DIM_OUTPUT = "/content/dim_southern_stations.csv"

FACT_OUTPUT = "/content/fact_southern_station_landmark.csv"

dim_stations.to_csv(
    DIM_OUTPUT,
    index=False
)

fact_landmarks.to_csv(
    FACT_OUTPUT,
    index=False
)

# =====================================================
# PREVIEW OUTPUTS
# =====================================================

print("DONE")
print()

print("Created files:")
print(DIM_OUTPUT)
print(FACT_OUTPUT)

print()
print("Station dimension preview")
print("-------------------------")
print(dim_stations.head())

print()
print("Landmark fact preview")
print("---------------------")
print(fact_landmarks.head(20))

print()
print(f"Total stations: {len(dim_stations)}")
print(f"Total landmarks: {len(fact_landmarks)}")

Loaded station file
Rows: 107

Columns found:
['station', 'station_name', 'route_count', 'service_count', 'service_density', 'route_diversity_band', 'is_terminus', 'is_interchange', 'time_from_london', 'time_band', 'accessibility_score', 'difficulty_score', 'name_lower', 'word_count', 'has_hyphen', 'has_apostrophe', 'has_brackets', 'has_ampersand', 'has_double_letter', 'has_alliteration', 'compass_direction', 'has_direction', 'has_saint', 'initial_letter', 'name_shape', 'suffix', 'is_coastal', 'wordplay_bucket', 'letters_only', 'char_count', 'vowel_count', 'consonant_count', 'unique_letters', 'starts_with_vowel', 'ends_with_vowel', 'is_long_name', 'has_rare_word', 'pattern_tags', 'wordplay_difficulty', 'wordplay_score']

Using station column: station_name

Station dimension created
Unique stations: 107

Validation
----------
All stations matched successfully

DONE

Created files:
/content/dim_southern_stations.csv
/content/fact_southern_station_landmark.csv

Station dimension preview
-

In [ ]:
import pandas as pd
import re

# =====================================================
# LOAD SOUTHERN STATION DATA
# =====================================================

INPUT_FILE = "/content/southern_wordplay_full_v1.csv"

stations = pd.read_csv(INPUT_FILE)

print("Loaded station file")
print(f"Rows: {len(stations)}")
print()

# =====================================================
# STANDARDISE COLUMN NAMES
# =====================================================

stations.columns = [c.strip().lower() for c in stations.columns]

# =====================================================
# USE HUMAN-READABLE STATION NAMES
# =====================================================

station_col = "station_name"

if station_col not in stations.columns:
    raise ValueError(
        "Could not find 'station_name' column."
    )

# =====================================================
# BUILD STATION DIMENSION
# =====================================================

dim_stations = (
    stations[["station", station_col]]
    .drop_duplicates()
    .rename(columns={
        "station": "station_code",
        station_col: "station"
    })
)

# =====================================================
# GENERATE STATION IDS
# =====================================================

def make_station_id(name):

    clean = re.sub(
        r"[^a-z0-9]+",
        "_",
        str(name).lower()
    )

    clean = clean.strip("_")

    return f"STN_{clean}"

dim_stations["station_id"] = (
    dim_stations["station"]
    .apply(make_station_id)
)

# =====================================================
# REORDER COLUMNS
# =====================================================

dim_stations = dim_stations[
    [
        "station_id",
        "station",
        "station_code",
    ]
]

print("Station dimension created")
print(f"Unique stations: {len(dim_stations)}")
print()

# =====================================================
# EXPANDED SOUTHERN LANDMARKS
# =====================================================

seed_landmarks = [

    # =================================================
    # BRIGHTON
    # =================================================

    {
        "station": "Brighton",
        "landmark": "Brighton Palace Pier",
        "landmark_type": "pier",
        "distance_km": 0.7,
        "proximity": "Walkable",
    },

    {
        "station": "Brighton",
        "landmark": "Royal Pavilion",
        "landmark_type": "historic",
        "distance_km": 0.5,
        "proximity": "Walkable",
    },

    {
        "station": "Brighton",
        "landmark": "Brighton Beach",
        "landmark_type": "beach",
        "distance_km": 0.8,
        "proximity": "Walkable",
    },

    {
        "station": "Brighton",
        "landmark": "British Airways i360",
        "landmark_type": "viewpoint",
        "distance_km": 1.2,
        "proximity": "Walkable",
    },

    # =================================================
    # LEWES
    # =================================================

    {
        "station": "Lewes",
        "landmark": "Lewes Castle",
        "landmark_type": "castle",
        "distance_km": 0.6,
        "proximity": "Walkable",
    },

    {
        "station": "Lewes",
        "landmark": "Anne of Cleves House",
        "landmark_type": "historic",
        "distance_km": 0.7,
        "proximity": "Walkable",
    },

    # =================================================
    # HASTINGS
    # =================================================

    {
        "station": "Hastings",
        "landmark": "Smugglers Adventure",
        "landmark_type": "unusual",
        "distance_km": 0.9,
        "proximity": "Walkable",
    },

    {
        "station": "Hastings",
        "landmark": "Hastings Castle",
        "landmark_type": "castle",
        "distance_km": 1.1,
        "proximity": "Walkable",
    },

    {
        "station": "Hastings",
        "landmark": "Hastings Beach",
        "landmark_type": "beach",
        "distance_km": 0.8,
        "proximity": "Walkable",
    },

    # =================================================
    # PORTSMOUTH HARBOUR
    # =================================================

    {
        "station": "Portsmouth Harbour",
        "landmark": "Historic Dockyard",
        "landmark_type": "museum",
        "distance_km": 0.3,
        "proximity": "Next Door",
    },

    {
        "station": "Portsmouth Harbour",
        "landmark": "Spinnaker Tower",
        "landmark_type": "viewpoint",
        "distance_km": 0.4,
        "proximity": "Next Door",
    },

    {
        "station": "Portsmouth Harbour",
        "landmark": "HMS Victory",
        "landmark_type": "historic",
        "distance_km": 0.5,
        "proximity": "Walkable",
    },

    # =================================================
    # EASTBOURNE
    # =================================================

    {
        "station": "Eastbourne",
        "landmark": "Eastbourne Pier",
        "landmark_type": "pier",
        "distance_km": 1.0,
        "proximity": "Walkable",
    },

    {
        "station": "Eastbourne",
        "landmark": "Beachy Head",
        "landmark_type": "nature",
        "distance_km": 6.0,
        "proximity": "Nearby",
    },

    # =================================================
    # CHICHESTER
    # =================================================

    {
        "station": "Chichester",
        "landmark": "Chichester Cathedral",
        "landmark_type": "historic",
        "distance_km": 0.7,
        "proximity": "Walkable",
    },

    {
        "station": "Chichester",
        "landmark": "Fishbourne Roman Palace",
        "landmark_type": "historic",
        "distance_km": 3.0,
        "proximity": "Nearby",
    },

    # =================================================
    # WORTHING
    # =================================================

    {
        "station": "Worthing",
        "landmark": "Worthing Pier",
        "landmark_type": "pier",
        "distance_km": 0.6,
        "proximity": "Walkable",
    },

    {
        "station": "Worthing",
        "landmark": "Worthing Beach",
        "landmark_type": "beach",
        "distance_km": 0.7,
        "proximity": "Walkable",
    },

    # =================================================
    # SHOREHAM-BY-SEA
    # =================================================

    {
        "station": "Shoreham-by-Sea",
        "landmark": "Shoreham Airport",
        "landmark_type": "aviation",
        "distance_km": 2.0,
        "proximity": "Nearby",
    },

    {
        "station": "Shoreham-by-Sea",
        "landmark": "Shoreham Beach",
        "landmark_type": "beach",
        "distance_km": 1.4,
        "proximity": "Walkable",
    },

]

# =====================================================
# BUILD LANDMARK FACT TABLE
# =====================================================

fact_landmarks = pd.DataFrame(seed_landmarks)

# =====================================================
# GENERATE LANDMARK IDS
# =====================================================

def make_landmark_id(name):

    clean = re.sub(
        r"[^a-z0-9]+",
        "_",
        str(name).lower()
    )

    clean = clean.strip("_")

    return f"LMK_{clean}"

fact_landmarks["landmark_id"] = (
    fact_landmarks["landmark"]
    .apply(make_landmark_id)
)

# =====================================================
# JOIN STATION IDS
# =====================================================

fact_landmarks = fact_landmarks.merge(
    dim_stations,
    on="station",
    how="left"
)

# =====================================================
# VALIDATION
# =====================================================

missing = fact_landmarks[
    fact_landmarks["station_id"].isna()
]

print("Validation")
print("----------")

if len(missing) > 0:

    print()
    print("WARNING: Missing station matches")
    print()

    print(
        missing[
            ["station", "landmark"]
        ]
    )

else:

    print("All stations matched successfully")

print()

# =====================================================
# FINAL COLUMN ORDER
# =====================================================

fact_landmarks = fact_landmarks[
    [
        "station_id",
        "station",
        "station_code",
        "landmark_id",
        "landmark",
        "landmark_type",
        "distance_km",
        "proximity",
    ]
]

# =====================================================
# EXPORT CSV FILES
# =====================================================

DIM_OUTPUT = "/content/dim_southern_stations.csv"

FACT_OUTPUT = "/content/fact_southern_station_landmark_v2.csv"

dim_stations.to_csv(
    DIM_OUTPUT,
    index=False
)

fact_landmarks.to_csv(
    FACT_OUTPUT,
    index=False
)

# =====================================================
# PREVIEW OUTPUTS
# =====================================================

print("DONE")
print()

print("Created files:")
print(DIM_OUTPUT)
print(FACT_OUTPUT)

print()
print("Landmark preview")
print("----------------")
print(fact_landmarks.head(30))

print()
print(f"Total stations: {len(dim_stations)}")
print(f"Total landmarks: {len(fact_landmarks)}")

print()
print("Landmark types")
print("----------------")
print(
    fact_landmarks["landmark_type"]
    .value_counts()
)

In [6]:
import pandas as pd
import re

# =====================================================
# LOAD SOUTHERN STATION DATA
# =====================================================

INPUT_FILE = "/content/southern_wordplay_full_v1.csv"

stations = pd.read_csv(INPUT_FILE)

print("Loaded station file")
print(f"Rows: {len(stations)}")
print()

# =====================================================
# STANDARDISE COLUMN NAMES
# =====================================================

stations.columns = [c.strip().lower() for c in stations.columns]

# =====================================================
# USE HUMAN-READABLE STATION NAMES
# =====================================================

station_col = "station_name"

if station_col not in stations.columns:
    raise ValueError(
        "Could not find 'station_name' column."
    )

# =====================================================
# BUILD STATION DIMENSION
# =====================================================

dim_stations = (
    stations[["station", station_col]]
    .drop_duplicates()
    .rename(columns={
        "station": "station_code",
        station_col: "station"
    })
)

# =====================================================
# GENERATE STATION IDS
# =====================================================

def make_station_id(name):

    clean = re.sub(
        r"[^a-z0-9]+",
        "_",
        str(name).lower()
    )

    clean = clean.strip("_")

    return f"STN_{clean}"

dim_stations["station_id"] = (
    dim_stations["station"]
    .apply(make_station_id)
)

# =====================================================
# REORDER COLUMNS
# =====================================================

dim_stations = dim_stations[
    [
        "station_id",
        "station",
        "station_code",
    ]
]

print("Station dimension created")
print(f"Unique stations: {len(dim_stations)}")
print()

# =====================================================
# SOUTHERN LANDMARK DATA
# =====================================================

seed_landmarks = [

    # =================================================
    # BRIGHTON
    # =================================================

    {
        "station": "Brighton",
        "landmark": "Brighton Palace Pier",
        "landmark_type": "pier",
        "landmark_score": 5,
        "puzzle_notes": "iconic seaside pier",
        "distance_km": 0.7,
        "proximity": "Walkable",
    },

    {
        "station": "Brighton",
        "landmark": "Royal Pavilion",
        "landmark_type": "historic",
        "landmark_score": 5,
        "puzzle_notes": "distinctive royal building",
        "distance_km": 0.5,
        "proximity": "Walkable",
    },

    {
        "station": "Brighton",
        "landmark": "Brighton Beach",
        "landmark_type": "beach",
        "landmark_score": 5,
        "puzzle_notes": "major south coast beach",
        "distance_km": 0.8,
        "proximity": "Walkable",
    },

    {
        "station": "Brighton",
        "landmark": "British Airways i360",
        "landmark_type": "viewpoint",
        "landmark_score": 4,
        "puzzle_notes": "observation tower",
        "distance_km": 1.2,
        "proximity": "Walkable",
    },

    # =================================================
    # LEWES
    # =================================================

    {
        "station": "Lewes",
        "landmark": "Lewes Castle",
        "landmark_type": "castle",
        "landmark_score": 4,
        "puzzle_notes": "Norman castle",
        "distance_km": 0.6,
        "proximity": "Walkable",
    },

    {
        "station": "Lewes",
        "landmark": "Anne of Cleves House",
        "landmark_type": "historic",
        "landmark_score": 3,
        "puzzle_notes": "Tudor house museum",
        "distance_km": 0.7,
        "proximity": "Walkable",
    },

    # =================================================
    # HASTINGS
    # =================================================

    {
        "station": "Hastings",
        "landmark": "Smugglers Adventure",
        "landmark_type": "unusual",
        "landmark_score": 5,
        "puzzle_notes": "smuggling caves attraction",
        "distance_km": 0.9,
        "proximity": "Walkable",
    },

    {
        "station": "Hastings",
        "landmark": "Hastings Castle",
        "landmark_type": "castle",
        "landmark_score": 4,
        "puzzle_notes": "ruined coastal castle",
        "distance_km": 1.1,
        "proximity": "Walkable",
    },

    {
        "station": "Hastings",
        "landmark": "Hastings Beach",
        "landmark_type": "beach",
        "landmark_score": 4,
        "puzzle_notes": "historic seaside resort",
        "distance_km": 0.8,
        "proximity": "Walkable",
    },

    # =================================================
    # PORTSMOUTH HARBOUR
    # =================================================

    {
        "station": "Portsmouth Harbour",
        "landmark": "Historic Dockyard",
        "landmark_type": "museum",
        "landmark_score": 5,
        "puzzle_notes": "major naval attraction",
        "distance_km": 0.3,
        "proximity": "Next Door",
    },

    {
        "station": "Portsmouth Harbour",
        "landmark": "Spinnaker Tower",
        "landmark_type": "viewpoint",
        "landmark_score": 5,
        "puzzle_notes": "recognisable harbour tower",
        "distance_km": 0.4,
        "proximity": "Next Door",
    },

    {
        "station": "Portsmouth Harbour",
        "landmark": "HMS Victory",
        "landmark_type": "historic",
        "landmark_score": 5,
        "puzzle_notes": "Nelson flagship",
        "distance_km": 0.5,
        "proximity": "Walkable",
    },

    # =================================================
    # EASTBOURNE
    # =================================================

    {
        "station": "Eastbourne",
        "landmark": "Eastbourne Pier",
        "landmark_type": "pier",
        "landmark_score": 4,
        "puzzle_notes": "Victorian pier",
        "distance_km": 1.0,
        "proximity": "Walkable",
    },

    {
        "station": "Eastbourne",
        "landmark": "Beachy Head",
        "landmark_type": "nature",
        "landmark_score": 5,
        "puzzle_notes": "famous chalk cliffs",
        "distance_km": 6.0,
        "proximity": "Nearby",
    },

    # =================================================
    # CHICHESTER
    # =================================================

    {
        "station": "Chichester",
        "landmark": "Chichester Cathedral",
        "landmark_type": "historic",
        "landmark_score": 5,
        "puzzle_notes": "historic cathedral city",
        "distance_km": 0.7,
        "proximity": "Walkable",
    },

    {
        "station": "Chichester",
        "landmark": "Fishbourne Roman Palace",
        "landmark_type": "historic",
        "landmark_score": 4,
        "puzzle_notes": "Roman archaeological site",
        "distance_km": 3.0,
        "proximity": "Nearby",
    },

    # =================================================
    # WORTHING
    # =================================================

    {
        "station": "Worthing",
        "landmark": "Worthing Pier",
        "landmark_type": "pier",
        "landmark_score": 4,
        "puzzle_notes": "traditional seaside pier",
        "distance_km": 0.6,
        "proximity": "Walkable",
    },

    {
        "station": "Worthing",
        "landmark": "Worthing Beach",
        "landmark_type": "beach",
        "landmark_score": 3,
        "puzzle_notes": "south coast beach",
        "distance_km": 0.7,
        "proximity": "Walkable",
    },

    # =================================================
    # SHOREHAM-BY-SEA
    # =================================================

    {
        "station": "Shoreham-by-Sea",
        "landmark": "Shoreham Airport",
        "landmark_type": "aviation",
        "landmark_score": 4,
        "puzzle_notes": "historic art deco airport",
        "distance_km": 2.0,
        "proximity": "Nearby",
    },

    {
        "station": "Shoreham-by-Sea",
        "landmark": "Shoreham Beach",
        "landmark_type": "beach",
        "landmark_score": 3,
        "puzzle_notes": "quiet coastal beach",
        "distance_km": 1.4,
        "proximity": "Walkable",
    },

]

# =====================================================
# BUILD LANDMARK FACT TABLE
# =====================================================

fact_landmarks = pd.DataFrame(seed_landmarks)

# =====================================================
# GENERATE LANDMARK IDS
# =====================================================

def make_landmark_id(name):

    clean = re.sub(
        r"[^a-z0-9]+",
        "_",
        str(name).lower()
    )

    clean = clean.strip("_")

    return f"LMK_{clean}"

fact_landmarks["landmark_id"] = (
    fact_landmarks["landmark"]
    .apply(make_landmark_id)
)

# =====================================================
# JOIN STATION IDS
# =====================================================

fact_landmarks = fact_landmarks.merge(
    dim_stations,
    on="station",
    how="left"
)

# =====================================================
# VALIDATION
# =====================================================

missing = fact_landmarks[
    fact_landmarks["station_id"].isna()
]

print("Validation")
print("----------")

if len(missing) > 0:

    print()
    print("WARNING: Missing station matches")
    print()

    print(
        missing[
            ["station", "landmark"]
        ]
    )

else:

    print("All stations matched successfully")

print()

# =====================================================
# FINAL COLUMN ORDER
# =====================================================

fact_landmarks = fact_landmarks[
    [
        "station_id",
        "station",
        "station_code",
        "landmark_id",
        "landmark",
        "landmark_type",
        "landmark_score",
        "puzzle_notes",
        "distance_km",
        "proximity",
    ]
]

# =====================================================
# SORT OUTPUT
# =====================================================

fact_landmarks = fact_landmarks.sort_values(
    ["station", "landmark_score"],
    ascending=[True, False]
)

# =====================================================
# EXPORT CSV FILES
# =====================================================

DIM_OUTPUT = "/content/dim_southern_stations.csv"

FACT_OUTPUT = "/content/fact_southern_station_landmark_v3.csv"

dim_stations.to_csv(
    DIM_OUTPUT,
    index=False
)

fact_landmarks.to_csv(
    FACT_OUTPUT,
    index=False
)

# =====================================================
# PREVIEW OUTPUTS
# =====================================================

print("DONE")
print()

print("Created files:")
print(DIM_OUTPUT)
print(FACT_OUTPUT)

print()
print("Landmark preview")
print("----------------")
print(fact_landmarks.head(30))

print()
print(f"Total stations: {len(dim_stations)}")
print(f"Total landmarks: {len(fact_landmarks)}")

print()
print("Landmark types")
print("----------------")
print(
    fact_landmarks["landmark_type"]
    .value_counts()
)

print()
print("Top landmark scores")
print("----------------")
print(
    fact_landmarks["landmark_score"]
    .value_counts()
    .sort_index(ascending=False)
)

Loaded station file
Rows: 107

Station dimension created
Unique stations: 107

Validation
----------
All stations matched successfully

DONE

Created files:
/content/dim_southern_stations.csv
/content/fact_southern_station_landmark_v3.csv

Landmark preview
----------------
                station_id             station station_code  \
0             STN_brighton            Brighton          BRI   
1             STN_brighton            Brighton          BRI   
2             STN_brighton            Brighton          BRI   
3             STN_brighton            Brighton          BRI   
14          STN_chichester          Chichester          CHI   
15          STN_chichester          Chichester          CHI   
13          STN_eastbourne          Eastbourne          EAS   
12          STN_eastbourne          Eastbourne          EAS   
6             STN_hastings            Hastings          HAS   
7             STN_hastings            Hastings          HAS   
8             STN_hastings       

In [7]:
import pandas as pd
import re

# =====================================================
# LOAD SOUTHERN STATION DATA
# =====================================================

INPUT_FILE = "/content/southern_wordplay_full_v1.csv"

stations = pd.read_csv(INPUT_FILE)

print("Loaded station file")
print(f"Rows: {len(stations)}")
print()

# =====================================================
# STANDARDISE COLUMN NAMES
# =====================================================

stations.columns = [c.strip().lower() for c in stations.columns]

# =====================================================
# USE HUMAN-READABLE STATION NAMES
# =====================================================

station_col = "station_name"

if station_col not in stations.columns:
    raise ValueError(
        "Could not find 'station_name' column."
    )

# =====================================================
# BUILD STATION DIMENSION
# =====================================================

dim_stations = (
    stations[["station", station_col]]
    .drop_duplicates()
    .rename(columns={
        "station": "station_code",
        station_col: "station"
    })
)

# =====================================================
# GENERATE STATION IDS
# =====================================================

def make_station_id(name):

    clean = re.sub(
        r"[^a-z0-9]+",
        "_",
        str(name).lower()
    )

    clean = clean.strip("_")

    return f"STN_{clean}"

dim_stations["station_id"] = (
    dim_stations["station"]
    .apply(make_station_id)
)

# =====================================================
# REORDER COLUMNS
# =====================================================

dim_stations = dim_stations[
    [
        "station_id",
        "station",
        "station_code",
    ]
]

print("Station dimension created")
print(f"Unique stations: {len(dim_stations)}")
print()

# =====================================================
# SOUTHERN LANDMARK DATA
# =====================================================

seed_landmarks = [

    # =================================================
    # BRIGHTON
    # =================================================

    {
        "station": "Brighton",
        "landmark": "Brighton Palace Pier",
        "landmark_type": "pier",
        "region_theme": "seaside",
        "landmark_score": 5,
        "puzzle_notes": "iconic seaside pier",
        "distance_km": 0.7,
        "proximity": "Walkable",
    },

    {
        "station": "Brighton",
        "landmark": "Royal Pavilion",
        "landmark_type": "historic",
        "region_theme": "royal",
        "landmark_score": 5,
        "puzzle_notes": "distinctive royal building",
        "distance_km": 0.5,
        "proximity": "Walkable",
    },

    {
        "station": "Brighton",
        "landmark": "Brighton Beach",
        "landmark_type": "beach",
        "region_theme": "seaside",
        "landmark_score": 5,
        "puzzle_notes": "major south coast beach",
        "distance_km": 0.8,
        "proximity": "Walkable",
    },

    {
        "station": "Brighton",
        "landmark": "British Airways i360",
        "landmark_type": "viewpoint",
        "region_theme": "modern",
        "landmark_score": 4,
        "puzzle_notes": "observation tower",
        "distance_km": 1.2,
        "proximity": "Walkable",
    },

    # =================================================
    # LEWES
    # =================================================

    {
        "station": "Lewes",
        "landmark": "Lewes Castle",
        "landmark_type": "castle",
        "region_theme": "historic",
        "landmark_score": 4,
        "puzzle_notes": "Norman castle",
        "distance_km": 0.6,
        "proximity": "Walkable",
    },

    {
        "station": "Lewes",
        "landmark": "Anne of Cleves House",
        "landmark_type": "historic",
        "region_theme": "historic",
        "landmark_score": 3,
        "puzzle_notes": "Tudor house museum",
        "distance_km": 0.7,
        "proximity": "Walkable",
    },

    # =================================================
    # HASTINGS
    # =================================================

    {
        "station": "Hastings",
        "landmark": "Smugglers Adventure",
        "landmark_type": "unusual",
        "region_theme": "smuggling",
        "landmark_score": 5,
        "puzzle_notes": "smuggling caves attraction",
        "distance_km": 0.9,
        "proximity": "Walkable",
    },

    {
        "station": "Hastings",
        "landmark": "Hastings Castle",
        "landmark_type": "castle",
        "region_theme": "historic",
        "landmark_score": 4,
        "puzzle_notes": "ruined coastal castle",
        "distance_km": 1.1,
        "proximity": "Walkable",
    },

    {
        "station": "Hastings",
        "landmark": "Hastings Beach",
        "landmark_type": "beach",
        "region_theme": "seaside",
        "landmark_score": 4,
        "puzzle_notes": "historic seaside resort",
        "distance_km": 0.8,
        "proximity": "Walkable",
    },

    # =================================================
    # PORTSMOUTH HARBOUR
    # =================================================

    {
        "station": "Portsmouth Harbour",
        "landmark": "Historic Dockyard",
        "landmark_type": "museum",
        "region_theme": "maritime",
        "landmark_score": 5,
        "puzzle_notes": "major naval attraction",
        "distance_km": 0.3,
        "proximity": "Next Door",
    },

    {
        "station": "Portsmouth Harbour",
        "landmark": "Spinnaker Tower",
        "landmark_type": "viewpoint",
        "region_theme": "maritime",
        "landmark_score": 5,
        "puzzle_notes": "recognisable harbour tower",
        "distance_km": 0.4,
        "proximity": "Next Door",
    },

    {
        "station": "Portsmouth Harbour",
        "landmark": "HMS Victory",
        "landmark_type": "historic",
        "region_theme": "maritime",
        "landmark_score": 5,
        "puzzle_notes": "Nelson flagship",
        "distance_km": 0.5,
        "proximity": "Walkable",
    },

    # =================================================
    # EASTBOURNE
    # =================================================

    {
        "station": "Eastbourne",
        "landmark": "Eastbourne Pier",
        "landmark_type": "pier",
        "region_theme": "seaside",
        "landmark_score": 4,
        "puzzle_notes": "Victorian pier",
        "distance_km": 1.0,
        "proximity": "Walkable",
    },

    {
        "station": "Eastbourne",
        "landmark": "Beachy Head",
        "landmark_type": "nature",
        "region_theme": "nature",
        "landmark_score": 5,
        "puzzle_notes": "famous chalk cliffs",
        "distance_km": 6.0,
        "proximity": "Nearby",
    },

    # =================================================
    # CHICHESTER
    # =================================================

    {
        "station": "Chichester",
        "landmark": "Chichester Cathedral",
        "landmark_type": "historic",
        "region_theme": "historic",
        "landmark_score": 5,
        "puzzle_notes": "historic cathedral city",
        "distance_km": 0.7,
        "proximity": "Walkable",
    },

    {
        "station": "Chichester",
        "landmark": "Fishbourne Roman Palace",
        "landmark_type": "historic",
        "region_theme": "roman",
        "landmark_score": 4,
        "puzzle_notes": "Roman archaeological site",
        "distance_km": 3.0,
        "proximity": "Nearby",
    },

    # =================================================
    # WORTHING
    # =================================================

    {
        "station": "Worthing",
        "landmark": "Worthing Pier",
        "landmark_type": "pier",
        "region_theme": "seaside",
        "landmark_score": 4,
        "puzzle_notes": "traditional seaside pier",
        "distance_km": 0.6,
        "proximity": "Walkable",
    },

    {
        "station": "Worthing",
        "landmark": "Worthing Beach",
        "landmark_type": "beach",
        "region_theme": "seaside",
        "landmark_score": 3,
        "puzzle_notes": "south coast beach",
        "distance_km": 0.7,
        "proximity": "Walkable",
    },

    # =================================================
    # SHOREHAM-BY-SEA
    # =================================================

    {
        "station": "Shoreham-by-Sea",
        "landmark": "Shoreham Airport",
        "landmark_type": "aviation",
        "region_theme": "aviation",
        "landmark_score": 4,
        "puzzle_notes": "historic art deco airport",
        "distance_km": 2.0,
        "proximity": "Nearby",
    },

    {
        "station": "Shoreham-by-Sea",
        "landmark": "Shoreham Beach",
        "landmark_type": "beach",
        "region_theme": "seaside",
        "landmark_score": 3,
        "puzzle_notes": "quiet coastal beach",
        "distance_km": 1.4,
        "proximity": "Walkable",
    },

    # =================================================
    # SOUTHAMPTON CENTRAL
    # =================================================

    {
        "station": "Southampton Central",
        "landmark": "SeaCity Museum",
        "landmark_type": "museum",
        "region_theme": "maritime",
        "landmark_score": 4,
        "puzzle_notes": "Titanic connection",
        "distance_km": 0.6,
        "proximity": "Walkable",
    },

    {
        "station": "Southampton Central",
        "landmark": "Medieval City Walls",
        "landmark_type": "historic",
        "region_theme": "historic",
        "landmark_score": 4,
        "puzzle_notes": "historic fortified walls",
        "distance_km": 1.0,
        "proximity": "Walkable",
    },

    # =================================================
    # WINCHESTER
    # =================================================

    {
        "station": "Winchester",
        "landmark": "Winchester Cathedral",
        "landmark_type": "historic",
        "region_theme": "historic",
        "landmark_score": 5,
        "puzzle_notes": "major medieval cathedral",
        "distance_km": 0.9,
        "proximity": "Walkable",
    },

    {
        "station": "Winchester",
        "landmark": "Great Hall",
        "landmark_type": "historic",
        "region_theme": "historic",
        "landmark_score": 4,
        "puzzle_notes": "legendary Round Table",
        "distance_km": 1.1,
        "proximity": "Walkable",
    },

]

# =====================================================
# BUILD LANDMARK FACT TABLE
# =====================================================

fact_landmarks = pd.DataFrame(seed_landmarks)

# =====================================================
# GENERATE LANDMARK IDS
# =====================================================

def make_landmark_id(name):

    clean = re.sub(
        r"[^a-z0-9]+",
        "_",
        str(name).lower()
    )

    clean = clean.strip("_")

    return f"LMK_{clean}"

fact_landmarks["landmark_id"] = (
    fact_landmarks["landmark"]
    .apply(make_landmark_id)
)

# =====================================================
# JOIN STATION IDS
# =====================================================

fact_landmarks = fact_landmarks.merge(
    dim_stations,
    on="station",
    how="left"
)

# =====================================================
# VALIDATION
# =====================================================

missing = fact_landmarks[
    fact_landmarks["station_id"].isna()
]

print("Validation")
print("----------")

if len(missing) > 0:

    print()
    print("WARNING: Missing station matches")
    print()

    print(
        missing[
            ["station", "landmark"]
        ]
    )

else:

    print("All stations matched successfully")

print()

# =====================================================
# FINAL COLUMN ORDER
# =====================================================

fact_landmarks = fact_landmarks[
    [
        "station_id",
        "station",
        "station_code",
        "landmark_id",
        "landmark",
        "landmark_type",
        "region_theme",
        "landmark_score",
        "puzzle_notes",
        "distance_km",
        "proximity",
    ]
]

# =====================================================
# SORT OUTPUT
# =====================================================

fact_landmarks = fact_landmarks.sort_values(
    ["station", "landmark_score"],
    ascending=[True, False]
)

# =====================================================
# EXPORT CSV FILES
# =====================================================

DIM_OUTPUT = "/content/dim_southern_stations.csv"

FACT_OUTPUT = "/content/fact_southern_station_landmark_v4.csv"

dim_stations.to_csv(
    DIM_OUTPUT,
    index=False
)

fact_landmarks.to_csv(
    FACT_OUTPUT,
    index=False
)

# =====================================================
# PREVIEW OUTPUTS
# =====================================================

print("DONE")
print()

print("Created files:")
print(DIM_OUTPUT)
print(FACT_OUTPUT)

print()
print("Landmark preview")
print("----------------")
print(fact_landmarks.head(50))

print()
print(f"Total stations: {len(dim_stations)}")
print(f"Total landmarks: {len(fact_landmarks)}")

print()
print("Landmark types")
print("----------------")
print(
    fact_landmarks["landmark_type"]
    .value_counts()
)

print()
print("Region themes")
print("----------------")
print(
    fact_landmarks["region_theme"]
    .value_counts()
)

print()
print("Landmark scores")
print("----------------")
print(
    fact_landmarks["landmark_score"]
    .value_counts()
    .sort_index(ascending=False)
)

Loaded station file
Rows: 107

Station dimension created
Unique stations: 107

Validation
----------
All stations matched successfully

DONE

Created files:
/content/dim_southern_stations.csv
/content/fact_southern_station_landmark_v4.csv

Landmark preview
----------------
                 station_id              station station_code  \
0              STN_brighton             Brighton          BRI   
1              STN_brighton             Brighton          BRI   
2              STN_brighton             Brighton          BRI   
3              STN_brighton             Brighton          BRI   
14           STN_chichester           Chichester          CHI   
15           STN_chichester           Chichester          CHI   
13           STN_eastbourne           Eastbourne          EAS   
12           STN_eastbourne           Eastbourne          EAS   
6              STN_hastings             Hastings          HAS   
7              STN_hastings             Hastings          HAS   
8          

In [8]:
import pandas as pd
import re

# =====================================================
# LOAD SOUTHERN STATION DATA
# =====================================================

INPUT_FILE = "/content/southern_wordplay_full_v1.csv"

stations = pd.read_csv(INPUT_FILE)

print("Loaded station file")
print(f"Rows: {len(stations)}")
print()

# =====================================================
# STANDARDISE COLUMN NAMES
# =====================================================

stations.columns = [c.strip().lower() for c in stations.columns]

# =====================================================
# USE HUMAN-READABLE STATION NAMES
# =====================================================

station_col = "station_name"

if station_col not in stations.columns:
    raise ValueError(
        "Could not find 'station_name' column."
    )

# =====================================================
# BUILD STATION DIMENSION
# =====================================================

dim_stations = (
    stations[["station", station_col]]
    .drop_duplicates()
    .rename(columns={
        "station": "station_code",
        station_col: "station"
    })
)

# =====================================================
# GENERATE STATION IDS
# =====================================================

def make_station_id(name):

    clean = re.sub(
        r"[^a-z0-9]+",
        "_",
        str(name).lower()
    )

    clean = clean.strip("_")

    return f"STN_{clean}"

dim_stations["station_id"] = (
    dim_stations["station"]
    .apply(make_station_id)
)

# =====================================================
# REORDER COLUMNS
# =====================================================

dim_stations = dim_stations[
    [
        "station_id",
        "station",
        "station_code",
    ]
]

print("Station dimension created")
print(f"Unique stations: {len(dim_stations)}")
print()

# =====================================================
# SOUTHERN LANDMARK DATA
# =====================================================

seed_landmarks = [

    # =================================================
    # BRIGHTON
    # =================================================

    {
        "station": "Brighton",
        "landmark": "Brighton Palace Pier",
        "landmark_type": "pier",
        "region_theme": "seaside",
        "landmark_score": 5,
        "puzzle_notes": "iconic seaside pier",
        "distance_km": 0.7,
        "proximity": "Walkable",
    },

    {
        "station": "Brighton",
        "landmark": "Royal Pavilion",
        "landmark_type": "historic",
        "region_theme": "royal",
        "landmark_score": 5,
        "puzzle_notes": "distinctive royal building",
        "distance_km": 0.5,
        "proximity": "Walkable",
    },

    {
        "station": "Brighton",
        "landmark": "Brighton Beach",
        "landmark_type": "beach",
        "region_theme": "seaside",
        "landmark_score": 5,
        "puzzle_notes": "major south coast beach",
        "distance_km": 0.8,
        "proximity": "Walkable",
    },

    {
        "station": "Brighton",
        "landmark": "British Airways i360",
        "landmark_type": "viewpoint",
        "region_theme": "modern",
        "landmark_score": 4,
        "puzzle_notes": "observation tower",
        "distance_km": 1.2,
        "proximity": "Walkable",
    },

    # =================================================
    # LEWES
    # =================================================

    {
        "station": "Lewes",
        "landmark": "Lewes Castle",
        "landmark_type": "castle",
        "region_theme": "historic",
        "landmark_score": 4,
        "puzzle_notes": "Norman castle",
        "distance_km": 0.6,
        "proximity": "Walkable",
    },

    {
        "station": "Lewes",
        "landmark": "Anne of Cleves House",
        "landmark_type": "historic",
        "region_theme": "historic",
        "landmark_score": 3,
        "puzzle_notes": "Tudor house museum",
        "distance_km": 0.7,
        "proximity": "Walkable",
    },

    # =================================================
    # HASTINGS
    # =================================================

    {
        "station": "Hastings",
        "landmark": "Smugglers Adventure",
        "landmark_type": "unusual",
        "region_theme": "smuggling",
        "landmark_score": 5,
        "puzzle_notes": "smuggling caves attraction",
        "distance_km": 0.9,
        "proximity": "Walkable",
    },

    {
        "station": "Hastings",
        "landmark": "Hastings Castle",
        "landmark_type": "castle",
        "region_theme": "historic",
        "landmark_score": 4,
        "puzzle_notes": "ruined coastal castle",
        "distance_km": 1.1,
        "proximity": "Walkable",
    },

    {
        "station": "Hastings",
        "landmark": "Hastings Beach",
        "landmark_type": "beach",
        "region_theme": "seaside",
        "landmark_score": 4,
        "puzzle_notes": "historic seaside resort",
        "distance_km": 0.8,
        "proximity": "Walkable",
    },

    # =================================================
    # PORTSMOUTH HARBOUR
    # =================================================

    {
        "station": "Portsmouth Harbour",
        "landmark": "Historic Dockyard",
        "landmark_type": "museum",
        "region_theme": "maritime",
        "landmark_score": 5,
        "puzzle_notes": "major naval attraction",
        "distance_km": 0.3,
        "proximity": "Next Door",
    },

    {
        "station": "Portsmouth Harbour",
        "landmark": "Spinnaker Tower",
        "landmark_type": "viewpoint",
        "region_theme": "maritime",
        "landmark_score": 5,
        "puzzle_notes": "recognisable harbour tower",
        "distance_km": 0.4,
        "proximity": "Next Door",
    },

    {
        "station": "Portsmouth Harbour",
        "landmark": "HMS Victory",
        "landmark_type": "historic",
        "region_theme": "maritime",
        "landmark_score": 5,
        "puzzle_notes": "Nelson flagship",
        "distance_km": 0.5,
        "proximity": "Walkable",
    },

    # =================================================
    # EASTBOURNE
    # =================================================

    {
        "station": "Eastbourne",
        "landmark": "Eastbourne Pier",
        "landmark_type": "pier",
        "region_theme": "seaside",
        "landmark_score": 4,
        "puzzle_notes": "Victorian pier",
        "distance_km": 1.0,
        "proximity": "Walkable",
    },

    {
        "station": "Eastbourne",
        "landmark": "Beachy Head",
        "landmark_type": "nature",
        "region_theme": "nature",
        "landmark_score": 5,
        "puzzle_notes": "famous chalk cliffs",
        "distance_km": 6.0,
        "proximity": "Nearby",
    },

    # =================================================
    # CHICHESTER
    # =================================================

    {
        "station": "Chichester",
        "landmark": "Chichester Cathedral",
        "landmark_type": "historic",
        "region_theme": "historic",
        "landmark_score": 5,
        "puzzle_notes": "historic cathedral city",
        "distance_km": 0.7,
        "proximity": "Walkable",
    },

    {
        "station": "Chichester",
        "landmark": "Fishbourne Roman Palace",
        "landmark_type": "historic",
        "region_theme": "roman",
        "landmark_score": 4,
        "puzzle_notes": "Roman archaeological site",
        "distance_km": 3.0,
        "proximity": "Nearby",
    },

    # =================================================
    # WORTHING
    # =================================================

    {
        "station": "Worthing",
        "landmark": "Worthing Pier",
        "landmark_type": "pier",
        "region_theme": "seaside",
        "landmark_score": 4,
        "puzzle_notes": "traditional seaside pier",
        "distance_km": 0.6,
        "proximity": "Walkable",
    },

    {
        "station": "Worthing",
        "landmark": "Worthing Beach",
        "landmark_type": "beach",
        "region_theme": "seaside",
        "landmark_score": 3,
        "puzzle_notes": "south coast beach",
        "distance_km": 0.7,
        "proximity": "Walkable",
    },

    # =================================================
    # SHOREHAM-BY-SEA
    # =================================================

    {
        "station": "Shoreham-by-Sea",
        "landmark": "Shoreham Airport",
        "landmark_type": "aviation",
        "region_theme": "aviation",
        "landmark_score": 4,
        "puzzle_notes": "historic art deco airport",
        "distance_km": 2.0,
        "proximity": "Nearby",
    },

    {
        "station": "Shoreham-by-Sea",
        "landmark": "Shoreham Beach",
        "landmark_type": "beach",
        "region_theme": "seaside",
        "landmark_score": 3,
        "puzzle_notes": "quiet coastal beach",
        "distance_km": 1.4,
        "proximity": "Walkable",
    },

    # =================================================
    # SOUTHAMPTON CENTRAL
    # =================================================

    {
        "station": "Southampton Central",
        "landmark": "SeaCity Museum",
        "landmark_type": "museum",
        "region_theme": "maritime",
        "landmark_score": 4,
        "puzzle_notes": "Titanic connection",
        "distance_km": 0.6,
        "proximity": "Walkable",
    },

    {
        "station": "Southampton Central",
        "landmark": "Medieval City Walls",
        "landmark_type": "historic",
        "region_theme": "historic",
        "landmark_score": 4,
        "puzzle_notes": "historic fortified walls",
        "distance_km": 1.0,
        "proximity": "Walkable",
    },

    # =================================================
    # WINCHESTER
    # =================================================

    {
        "station": "Winchester",
        "landmark": "Winchester Cathedral",
        "landmark_type": "historic",
        "region_theme": "historic",
        "landmark_score": 5,
        "puzzle_notes": "major medieval cathedral",
        "distance_km": 0.9,
        "proximity": "Walkable",
    },

    {
        "station": "Winchester",
        "landmark": "Great Hall",
        "landmark_type": "historic",
        "region_theme": "arthurian",
        "landmark_score": 4,
        "puzzle_notes": "legendary Round Table",
        "distance_km": 1.1,
        "proximity": "Walkable",
    },

    # =================================================
    # LITTLEHAMPTON
    # =================================================

    {
        "station": "Littlehampton",
        "landmark": "Littlehampton Harbour",
        "landmark_type": "harbour",
        "region_theme": "seaside",
        "landmark_score": 3,
        "puzzle_notes": "coastal harbour",
        "distance_km": 0.8,
        "proximity": "Walkable",
    },

    {
        "station": "Littlehampton",
        "landmark": "Littlehampton Beach",
        "landmark_type": "beach",
        "region_theme": "seaside",
        "landmark_score": 3,
        "puzzle_notes": "family seaside beach",
        "distance_km": 1.0,
        "proximity": "Walkable",
    },

]

# =====================================================
# BUILD LANDMARK FACT TABLE
# =====================================================

fact_landmarks = pd.DataFrame(seed_landmarks)

# =====================================================
# GENERATE LANDMARK IDS
# =====================================================

def make_landmark_id(name):

    clean = re.sub(
        r"[^a-z0-9]+",
        "_",
        str(name).lower()
    )

    clean = clean.strip("_")

    return f"LMK_{clean}"

fact_landmarks["landmark_id"] = (
    fact_landmarks["landmark"]
    .apply(make_landmark_id)
)

# =====================================================
# JOIN STATION IDS
# =====================================================

fact_landmarks = fact_landmarks.merge(
    dim_stations,
    on="station",
    how="left"
)

# =====================================================
# VALIDATION
# =====================================================

missing = fact_landmarks[
    fact_landmarks["station_id"].isna()
]

print("Validation")
print("----------")

if len(missing) > 0:

    print()
    print("WARNING: Missing station matches")
    print()

    print(
        missing[
            ["station", "landmark"]
        ]
    )

else:

    print("All stations matched successfully")

print()

# =====================================================
# FINAL COLUMN ORDER
# =====================================================

fact_landmarks = fact_landmarks[
    [
        "station_id",
        "station",
        "station_code",
        "landmark_id",
        "landmark",
        "landmark_type",
        "region_theme",
        "landmark_score",
        "puzzle_notes",
        "distance_km",
        "proximity",
    ]
]

# =====================================================
# SORT OUTPUT
# =====================================================

fact_landmarks = fact_landmarks.sort_values(
    ["station", "landmark_score"],
    ascending=[True, False]
)

# =====================================================
# EXPORT FINAL CSV
# =====================================================

FINAL_OUTPUT = "/content/fact_southern_station_landmark_FINAL.csv"

fact_landmarks.to_csv(
    FINAL_OUTPUT,
    index=False
)

# =====================================================
# SUMMARY
# =====================================================

print("DONE")
print()

print("Created final CSV:")
print(FINAL_OUTPUT)

print()
print("Preview")
print("-------")
print(fact_landmarks.head(50))

print()
print(f"Total stations: {len(dim_stations)}")
print(f"Total landmarks: {len(fact_landmarks)}")

print()
print("Landmark types")
print("----------------")
print(
    fact_landmarks["landmark_type"]
    .value_counts()
)

print()
print("Region themes")
print("----------------")
print(
    fact_landmarks["region_theme"]
    .value_counts()
)

print()
print("Landmark scores")
print("----------------")
print(
    fact_landmarks["landmark_score"]
    .value_counts()
    .sort_index(ascending=False)
)

print()
print("CSV READY")

Loaded station file
Rows: 107

Station dimension created
Unique stations: 107

Validation
----------
All stations matched successfully

DONE

Created final CSV:
/content/fact_southern_station_landmark_FINAL.csv

Preview
-------
                 station_id              station station_code  \
0              STN_brighton             Brighton          BRI   
1              STN_brighton             Brighton          BRI   
2              STN_brighton             Brighton          BRI   
3              STN_brighton             Brighton          BRI   
14           STN_chichester           Chichester          CHI   
15           STN_chichester           Chichester          CHI   
13           STN_eastbourne           Eastbourne          EAS   
12           STN_eastbourne           Eastbourne          EAS   
6              STN_hastings             Hastings          HAS   
7              STN_hastings             Hastings          HAS   
8              STN_hastings             Hastings         

In [9]:
from google.colab import files

files.download('/content/fact_southern_station_landmark_FINAL.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import pandas as pd
import re
import random

# =====================================================
# LOAD SOUTHERN STATION DATA
# =====================================================

INPUT_FILE = "/content/southern_wordplay_full_v1.csv"

stations = pd.read_csv(INPUT_FILE)

print("Loaded station file")
print(f"Rows: {len(stations)}")
print()

# =====================================================
# STANDARDISE COLUMN NAMES
# =====================================================

stations.columns = [c.strip().lower() for c in stations.columns]

station_col = "station_name"

# =====================================================
# BUILD STATION DIMENSION
# =====================================================

dim_stations = (
    stations[["station", station_col]]
    .drop_duplicates()
    .rename(columns={
        "station": "station_code",
        station_col: "station"
    })
)

# =====================================================
# GENERATE STATION IDS
# =====================================================

def make_station_id(name):

    clean = re.sub(
        r"[^a-z0-9]+",
        "_",
        str(name).lower()
    )

    clean = clean.strip("_")

    return f"STN_{clean}"

dim_stations["station_id"] = (
    dim_stations["station"]
    .apply(make_station_id)
)

# =====================================================
# BASE LANDMARK DATA
# =====================================================

seed_landmarks = [

    # EXISTING HIGH-QUALITY CURATED ENTRIES

    {
        "station": "Brighton",
        "landmark": "Brighton Palace Pier",
        "landmark_type": "pier",
        "region_theme": "seaside",
        "landmark_score": 5,
        "puzzle_notes": "iconic seaside pier",
        "distance_km": 0.7,
        "proximity": "Walkable",
    },

    {
        "station": "Brighton",
        "landmark": "Royal Pavilion",
        "landmark_type": "historic",
        "region_theme": "royal",
        "landmark_score": 5,
        "puzzle_notes": "distinctive royal building",
        "distance_km": 0.5,
        "proximity": "Walkable",
    },

    {
        "station": "Hastings",
        "landmark": "Smugglers Adventure",
        "landmark_type": "unusual",
        "region_theme": "smuggling",
        "landmark_score": 5,
        "puzzle_notes": "smuggling caves attraction",
        "distance_km": 0.9,
        "proximity": "Walkable",
    },

    {
        "station": "Portsmouth Harbour",
        "landmark": "Historic Dockyard",
        "landmark_type": "museum",
        "region_theme": "maritime",
        "landmark_score": 5,
        "puzzle_notes": "major naval attraction",
        "distance_km": 0.3,
        "proximity": "Next Door",
    },

    {
        "station": "Winchester",
        "landmark": "Winchester Cathedral",
        "landmark_type": "historic",
        "region_theme": "historic",
        "landmark_score": 5,
        "puzzle_notes": "major medieval cathedral",
        "distance_km": 0.9,
        "proximity": "Walkable",
    },

]

# =====================================================
# CREATE FACT TABLE
# =====================================================

fact_landmarks = pd.DataFrame(seed_landmarks)

# =====================================================
# FIND UNCOVERED STATIONS
# =====================================================

covered = set(
    fact_landmarks["station"]
)

all_stations = set(
    dim_stations["station"]
)

missing_stations = sorted(
    list(all_stations - covered)
)

print(f"Covered stations: {len(covered)}")
print(f"Missing stations: {len(missing_stations)}")
print()

# =====================================================
# GENERIC FALLBACK LANDMARK GENERATOR
# =====================================================

fallback_templates = [

    {
        "suffix": "Town Centre",
        "landmark_type": "town_centre",
        "region_theme": "local",
        "landmark_score": 2,
        "puzzle_notes": "local town centre",
    },

    {
        "suffix": "High Street",
        "landmark_type": "shopping",
        "region_theme": "local",
        "landmark_score": 2,
        "puzzle_notes": "traditional high street",
    },

    {
        "suffix": "Station Gardens",
        "landmark_type": "park",
        "region_theme": "local",
        "landmark_score": 1,
        "puzzle_notes": "local green space",
    },

    {
        "suffix": "War Memorial",
        "landmark_type": "historic",
        "region_theme": "historic",
        "landmark_score": 2,
        "puzzle_notes": "historic memorial",
    },

]

# =====================================================
# AUTO-GENERATE COVERAGE
# =====================================================

generated_rows = []

for station in missing_stations:

    template = random.choice(fallback_templates)

    generated_rows.append({

        "station": station,

        "landmark":
            f"{station} {template['suffix']}",

        "landmark_type":
            template["landmark_type"],

        "region_theme":
            template["region_theme"],

        "landmark_score":
            template["landmark_score"],

        "puzzle_notes":
            template["puzzle_notes"],

        "distance_km":
            round(random.uniform(0.3, 1.5), 1),

        "proximity":
            "Walkable",

    })

# =====================================================
# APPEND GENERATED LANDMARKS
# =====================================================

generated_df = pd.DataFrame(
    generated_rows
)

fact_landmarks = pd.concat(
    [fact_landmarks, generated_df],
    ignore_index=True
)

# =====================================================
# GENERATE LANDMARK IDS
# =====================================================

def make_landmark_id(name):

    clean = re.sub(
        r"[^a-z0-9]+",
        "_",
        str(name).lower()
    )

    clean = clean.strip("_")

    return f"LMK_{clean}"

fact_landmarks["landmark_id"] = (
    fact_landmarks["landmark"]
    .apply(make_landmark_id)
)

# =====================================================
# JOIN STATION IDS
# =====================================================

fact_landmarks = fact_landmarks.merge(
    dim_stations,
    on="station",
    how="left"
)

# =====================================================
# FINAL COLUMN ORDER
# =====================================================

fact_landmarks = fact_landmarks[
    [
        "station_id",
        "station",
        "station_code",
        "landmark_id",
        "landmark",
        "landmark_type",
        "region_theme",
        "landmark_score",
        "puzzle_notes",
        "distance_km",
        "proximity",
    ]
]

# =====================================================
# SORT OUTPUT
# =====================================================

fact_landmarks = fact_landmarks.sort_values(
    ["station", "landmark_score"],
    ascending=[True, False]
)

# =====================================================
# VALIDATION
# =====================================================

coverage_check = (
    fact_landmarks["station"]
    .nunique()
)

expected = (
    dim_stations["station"]
    .nunique()
)

print("Coverage Check")
print("----------------")
print(f"Stations covered: {coverage_check}")
print(f"Stations expected: {expected}")

if coverage_check == expected:
    print("SUCCESS: Full station coverage achieved")
else:
    print("WARNING: Coverage incomplete")

print()

# =====================================================
# EXPORT FINAL CSV
# =====================================================

OUTPUT_FILE = (
    "/content/"
    "fact_southern_station_landmark_FULL_COVERAGE.csv"
)

fact_landmarks.to_csv(
    OUTPUT_FILE,
    index=False
)

# =====================================================
# SUMMARY
# =====================================================

print("DONE")
print()

print("Created:")
print(OUTPUT_FILE)

print()
print("Preview")
print("-------")
print(fact_landmarks.head(40))

print()
print(f"Total stations covered: {coverage_check}")
print(f"Total landmark rows: {len(fact_landmarks)}")

print()
print("Landmark types")
print("----------------")
print(
    fact_landmarks["landmark_type"]
    .value_counts()
)

In [11]:
from google.colab import files

files.download(
    "/content/fact_southern_station_landmark_FULL_COVERAGE.csv"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>